# 🎙️ PIPELINE TỰ ĐỘNG SINH ÂM THANH 120 CÂU HUIT-AUDIO CHO BÀI TOÁN ASR / STT
---
**Mục tiêu:**
- Tận dụng kiến trúc **VieNeu-TTS-v3-Turbo** từ file `TTS.ipynb` trên Google Drive của bạn.
- **Bỏ qua câu 1 đến 30:** Dành riêng cho **Người thật đọc (Human Voice)**.
- **Từ câu 31 đến 120:** Tự động sinh audio bằng **VieNeu-TTS-v3**, luân phiên **10 câu một vùng miền / giới tính** khác nhau (Bắc - Trung - Nam, Nam/Nữ).
- Tự động ép chuẩn **16 kHz Mono WAV** (chuẩn đầu vào của Whisper / PhoWhisper).
- Tự động cập nhật metadata vùng miền, giới tính vào file `HUIT_Audio_120_Queries_Updated.xlsx` và nén file zip để tải về.

In [ ]:
# CELL 1: Cài đặt thư viện & Mount Google Drive
!pip install -q vieneu pydub soundfile openpyxl pandas

from google.colab import drive
drive.mount('/content/drive')
print('✅ Đã kết nối Google Drive!')

In [ ]:
# CELL 2: Khởi tạo Module VieNeu TTS từ Google Drive (Kế thừa từ TTS.ipynb)
import os
import soundfile as sf
from vieneu import Vieneu
from pydub import AudioSegment

# Đường dẫn tới thư mục model VieNeu đã lưu trên Drive của bạn
MODEL_DRIVE_PATH = "/content/drive/MyDrive/models/vieneu-tts-v3-turbo"

# Thiết lập đường dẫn offline chuẩn cho HuggingFace
os.environ["HF_HOME"] = MODEL_DRIVE_PATH

print(f"[Module TTS] Đang nạp model VieNeu từ: {MODEL_DRIVE_PATH}")
tts = Vieneu()

# Lấy danh sách preset voice có sẵn
preset_voices = tts.list_preset_voices()
print(f"✅ Đã nạp thành công! Có {len(preset_voices)} giọng đọc trong VieNeu:")
for idx, (desc, vid) in enumerate(preset_voices):
    print(f"  [{idx:02d}] Voice ID: '{vid}' | Mô tả: {desc}")

In [ ]:
# CELL 3: Cấu hình phân chia vùng miền (10 câu / vùng miền từ câu 31 đến 120)
import pandas as pd

# Bạn có thể kiểm tra danh sách in ra ở Cell 2 để gán index_voice tương ứng
# Nếu danh sách có ít voice, hệ thống sẽ tự động dùng modulo an toàn không bao giờ bị lỗi index!
total_v = len(preset_voices)

VOICE_BLOCKS = [
    {"start": 31,  "end": 40,  "region": "Nam",   "gender": "Nữ",  "voice_idx": 1 % total_v, "note": "Nữ miền Nam"},
    {"start": 41,  "end": 50,  "region": "Nam",   "gender": "Nam", "voice_idx": 0 % total_v, "note": "Nam miền Nam"},
    {"start": 51,  "end": 60,  "region": "Bắc",   "gender": "Nữ",  "voice_idx": 2 % total_v, "note": "Nữ miền Bắc"},
    {"start": 61,  "end": 70,  "region": "Bắc",   "gender": "Nam", "voice_idx": 3 % total_v, "note": "Nam miền Bắc"},
    {"start": 71,  "end": 80,  "region": "Trung", "gender": "Nam", "voice_idx": 4 % total_v, "note": "Nam miền Trung"},
    {"start": 81,  "end": 90,  "region": "Trung", "gender": "Nữ",  "voice_idx": 5 % total_v, "note": "Nữ miền Trung"},
    {"start": 91,  "end": 100, "region": "Nam",   "gender": "Nữ",  "voice_idx": 6 % total_v, "note": "Nữ miền Nam (Trẻ)"},
    {"start": 101, "end": 110, "region": "Bắc",   "gender": "Nữ",  "voice_idx": 7 % total_v, "note": "Nữ miền Bắc (Truyền cảm)"},
    {"start": 111, "end": 120, "region": "Bắc",   "gender": "Nam", "voice_idx": 8 % total_v, "note": "Nam miền Bắc (Trầm ấm)"},
]

print("✅ Đã cấu hình xong 9 block vùng miền cho các câu từ 31 đến 120!")

### 🎙️ CELL: TỰ ĐỘNG CHUẨN HOÁ 30 FILE ÂM THANH CỦA BẠN BẠN TRÊN COLAB
**Cách dùng siêu đơn giản:**
1. Nén 30 file của bạn bạn thành 1 file zip (ví dụ: `30_files.zip`) rồi upload lên Colab (ở cột bên trái).
2. Chạy cell dưới đây: Script sẽ tự động giải nén, sắp xếp thứ tự, chuyển về đúng chuẩn **16.000 Hz Mono WAV**, và đổi tên thành `huit_001.wav` đến `huit_030.wav` đặt thẳng vào thư mục `huit_audio_16k_wav`!

In [ ]:
# CÀI ĐẶT & CHUẨN HOÁ 30 FILE CỦA NGƯỜI THẬT
!pip install -q pydub

import os, zipfile, glob
from pydub import AudioSegment

raw_input_dir = "raw_friend_audios"
os.makedirs(raw_input_dir, exist_ok=True)

# Tìm file zip upload lên (bỏ qua file zip output)
zip_candidates = [z for z in glob.glob("*.zip") if "huit_audio_synthetic" not in z]

if zip_candidates:
    zip_to_extract = zip_candidates[0]
    print(f"📦 Đang giải nén: {zip_to_extract}...")
    with zipfile.ZipFile(zip_to_extract, "r") as zip_ref:
        zip_ref.extractall(raw_input_dir)
else:
    print("ℹ️ Nếu bạn không nén zip, hãy tạo thư mục 'raw_friend_audios' và tải thẳng 30 file vào đó.")

output_dir = "huit_audio_16k_wav"
os.makedirs(output_dir, exist_ok=True)

valid_exts = (".wav", ".mp3", ".m4a", ".aac", ".mp4", ".ogg")
raw_files = []
for root, dirs, files in os.walk(raw_input_dir):
    for f in files:
        if f.lower().endswith(valid_exts):
            raw_files.append(os.path.join(root, f))

raw_files = sorted(raw_files)
print(f"\n🔍 Tìm thấy {len(raw_files)} file. Đang chuẩn hoá về 16kHz Mono WAV...")

for i, in_path in enumerate(raw_files[:30], 1):
    audio = AudioSegment.from_file(in_path)
    # Ép chuẩn Whisper: 16000Hz, 1 kênh Mono, 16-bit PCM
    audio = audio.set_frame_rate(16000).set_channels(1).set_sample_width(2)
    
    out_name = f"huit_{i:03d}.wav"
    out_path = os.path.join(output_dir, out_name)
    audio.export(out_path, format="wav")
    print(f"[{i:02d}/30] {os.path.basename(in_path)} -> {out_name} (16kHz Mono)")

print(f"\n🎉 XONG! Đã chuẩn hoá {min(len(raw_files), 30)} file người thật vào thư mục '{output_dir}'.")

# Nén thư mục thành file zip để tải về máy ngay
!zip -q -r huit_audio_30_human.zip huit_audio_16k_wav
print("📦 ĐÃ NÉN XONG! File zip tải về: huit_audio_30_human.zip")

from google.colab import files
files.download("huit_audio_30_human.zip")


In [ ]:
# CELL 4: Chạy sinh toàn bộ file Audio và chuẩn hoá về 16kHz Mono WAV cho Whisper
output_dir = "huit_audio_16k_wav"
os.makedirs(output_dir, exist_ok=True)

excel_file = "HUIT_Audio_120_Queries.xlsx"
if not os.path.exists(excel_file):
    raise FileNotFoundError(f"Vui lòng upload file '{excel_file}' lên Colab trước khi chạy!")

df = pd.read_excel(excel_file)
print(f"Đã đọc {len(df)} câu hỏi từ file Excel.\nBắt đầu xử lý:\n - Câu 01-30: Giữ cho Người thật đọc\n - Câu 31-120: Tự động sinh VieNeu-TTS\n")

for idx, row in df.iterrows():
    stt = int(row["STT"])
    text = str(row["Nội Dung Cần Đọc (Transcript)"])
    wav_filename = str(row["Tên File WAV"])
    
    # 1. Bỏ qua câu 1 đến 30 (dành cho bạn của bạn đọc)
    if stt <= 30:
        df.at[idx, "Giới Tính (Nam/Nữ)"] = "Theo người đọc thật"
        df.at[idx, "Vùng Miền (Bắc/Trung/Nam)"] = "Theo người đọc thật"
        df.at[idx, "Ghi Chú"] = "Human Voice (Người thật đọc)"
        continue

    # 2. Tìm cấu hình vùng miền của câu hiện tại
    cfg = next((b for b in VOICE_BLOCKS if b["start"] <= stt <= b["end"]), None)
    if cfg is None:
        continue
        
    voice_idx = cfg["voice_idx"]
    voice_name = preset_voices[voice_idx][1]
    region = cfg["region"]
    gender = cfg["gender"]
    note = cfg["note"]
    
    # Cập nhật metadata vào DataFrame
    df.at[idx, "Giới Tính (Nam/Nữ)"] = gender
    df.at[idx, "Vùng Miền (Bắc/Trung/Nam)"] = region
    df.at[idx, "Ghi Chú"] = f"VieNeu ({note}) - {voice_name}"

    # 3. Lấy voice và sinh âm thanh qua VieNeu
    selected_voice = tts.get_preset_voice(voice_name)
    temp_wav = f"temp_{stt}.wav"
    audio_data = tts.infer(text=text, voice=selected_voice)
    tts.save(audio_data, temp_wav)
    
    # 4. Ép chuẩn 16 kHz Mono WAV cho Whisper / PhoWhisper
    final_path = os.path.join(output_dir, wav_filename)
    sound = AudioSegment.from_file(temp_wav)
    sound = sound.set_frame_rate(16000).set_channels(1).set_sample_width(2)
    sound.export(final_path, format="wav")
    
    if os.path.exists(temp_wav):
        os.remove(temp_wav)
        
    print(f"[{stt:03d}/120] -> Tạo {wav_filename} | Miền {region} ({gender}) | Voice: {voice_name}")

# Lưu file Excel mới đã được điền đầy đủ metadata
updated_excel = "HUIT_Audio_120_Queries_Updated.xlsx"
df.to_excel(updated_excel, index=False)
print(f"\n🎉 HOÀN THÀNH TẤT CẢ! Đã cập nhật metadata vào: {updated_excel}")

# Nén thư mục audio thành file zip để tải về
!zip -q -r huit_audio_synthetic_31_120.zip huit_audio_16k_wav
print("📦 Đã tạo file zip: huit_audio_synthetic_31_120.zip (Sẵn sàng tải về!)")

In [ ]:
# CELL 5: Nghe thử một câu mẫu vừa tạo ngay trên Colab
from IPython.display import Audio, display

sample_test = "huit_audio_16k_wav/huit_035.wav"
if os.path.exists(sample_test):
    print(f"Nghe thử câu số 35: {sample_test}")
    display(Audio(sample_test))
else:
    print("Chưa có file audio để phát thử.")